<a href="https://colab.research.google.com/github/kokilahemachandra-afk/bolivia-sdg-explorer/blob/main/Bolivia_SDG_13_and_15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Step 1: Hypotheses

which indicators should be connected, before looking at the data

**SDG 13: Climate action**


```
* sdg13_1_fec: families affected by climate disasters (%)
* sdg13_1_icc: climate vulnerability index
* sdg13_2_epb: CO₂ from forest loss (t per person)
* sdg13_2_ect: CO₂ from transport (t per person)
* sdg13_2_eci: CO₂ from fires (t per person)
```



**SDG 15: Life on land**

	* sdg15_1_sap: protected area (% of land)
	* sdg15_2_tpb: forest loss rate (% of forest per year)
	* sdg15_5_pbd: biodiversity loss from deforestation
	* sdg15_5_pbi: biodiversity loss from fires

**Hypothesis and Reason**

1. H1	sdg13_2_eci (CO₂ from fires) ↔ sdg15_5_pbi (biodiversity loss from fires) :
strong positive	Same fires, same burned area

2. H2	sdg13_2_epb (CO₂ from forest loss) ↔ sdg15_2_tpb (forest loss rate) and sdg15_5_pbd:
positive	Same deforestation

3. H3	sdg13_2_ect (transport CO₂): unrelated to SDG 15	Cars and trucks have nothing to do with forests

4. H4	sdg15_1_sap (protected area) ↔ forest-loss CO₂: negative?	Protection should stop clearing… unless protected areas sit where the forests (and the pressure) are

## Step 2: Load and merge

**Line by line:**

* `geo.load("sdg13") `without variables= loads the index and all 5 components this time.
* `.merge(..., on=["geo_id", "year"])` joins the two tables. Rows are matched when both the municipality and the year are the same, so Azurduy‑2012 from SDG 13 is paired with Azurduy‑2012 from SDG 15.
* `.columns.tolist()` lists every column name.

In [ ]:
!pip install -q geo4bolivia

In [ ]:
import geo4bolivia as geo
import pandas as pd

sdg13 = geo.load("sdg13")          # no 'variables=' → all components
sdg15 = geo.load("sdg15")

both = sdg13.merge(sdg15, on=["geo_id", "year"])

print(sdg13.shape, sdg15.shape, both.shape)
print(both.columns.tolist())

## Step 3: Clean up and check missing data

In [ ]:
# 1. Remove the duplicate ID column
both = both.drop(columns="mun_id_y").rename(columns={"mun_id_x": "mun_id"})

# 2. Count missing values per column
print(both.isna().sum())

# 3. Mark municipalities that have forest data
both["forest"] = both["sdg15_2_tpb"].notna()

# 4. Compare the average index for the two groups
print(both.groupby(["year", "forest"])[["index_sdg13", "index_sdg15"]].mean().round(1))

## Step 5: PCA

**Four decisions before PCA**

* The 9 components only, not the two indices -> Each index is an average of its components, so including it would count the same information twice
* Keep only the 190 municipalities with forest data -> PCA cannot handle empty cells. (This is where Step 3 pays off.)
* One year - 2024 -> Each municipality should appear only once
* Rescale each variable to mean 0, SD 1 -> Units differ (% vs tonnes vs an index). Without this, the variable with the biggest numbers would dominate





In [ ]:
import plotly.express as px

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

comp = ["sdg13_1_fec", "sdg13_1_icc", "sdg13_2_epb", "sdg13_2_ect", "sdg13_2_eci",
        "sdg15_1_sap", "sdg15_2_tpb", "sdg15_5_pbd", "sdg15_5_pbi"]

# Decisions 2 + 3: one year, complete cases only
d = both[both["year"] == 2024].dropna(subset=comp)
print("Municipalities used:", len(d))

# Decision 4: standardise
Z = StandardScaler().fit_transform(d[comp])

# Run PCA
pca = PCA().fit(Z)

# (a) How much information each component captures
print(pd.Series(pca.explained_variance_ratio_.round(2),
                index=[f"PC{i+1}" for i in range(9)]))

# (b) Loadings: how strongly each indicator belongs to each component
loadings = pd.DataFrame(pca.components_[:3].T, index=comp,
                        columns=["PC1", "PC2", "PC3"]).round(2)

print(loadings)


labels = {"sdg13_1_fec": "13: Families affected by disasters",
          "sdg13_1_icc": "13: Climate vulnerability",
          "sdg13_2_epb": "13: CO₂ from forest loss",
          "sdg13_2_ect": "13: CO₂ from transport",
          "sdg13_2_eci": "13: CO₂ from fires",
          "sdg15_1_sap": "15: Protected area",
          "sdg15_2_tpb": "15: Forest loss rate",
          "sdg15_5_pbd": "15: Biodiversity loss (deforestation)",
          "sdg15_5_pbi": "15: Biodiversity loss (fires)"}


px.imshow(loadings, text_auto=True, color_continuous_scale="RdBu",
          zmin=-0.6, zmax=0.6, aspect="auto",
          title="PCA loadings: SDG 13 + SDG 15 indicators (~2024)").show()

## Step 6: Robustness check (rank-based PCA)

test whether a few extreme municipalities are driving them. We replace each value by its rank (1 = lowest, 190 = highest). Ranks ignore how extreme the outliers are



In [ ]:
import plotly.express as px

Zr = StandardScaler().fit_transform(d[comp].rank())
pca_r = PCA().fit(Zr)

print(pd.Series(pca_r.explained_variance_ratio_.round(2),
                index=[f"PC{i+1}" for i in range(9)]))

loadings_r = pd.DataFrame(pca_r.components_[:3].T, index=comp,
                          columns=["PC1", "PC2", "PC3"]).round(2)
print(loadings_r)

## Step 7: Where are these patterns? (map the PC scores)

So far we know which indicators go together. Now we ask which municipalities score high. Each municipality gets a score on each PC: a high PC1 score means strong fire and forest-loss pressure.

In [ ]:
import geo4bolivia as geo

# 1. Score each municipality on PC1-PC3
scores = pd.DataFrame(pca_r.transform(Zr)[:, :3],
                      columns=["PC1", "PC2", "PC3"],
                      index=d["geo_id"]).round(2)

# 2. Get municipality boundaries and attach the scores
gdf = geo.load("sdg13", variables="index_sdg13", years=2024, geometry=True)
gdf = gdf.merge(scores, left_on="geo_id", right_index=True, how="left")
gdf = gdf.set_index("geo_id")

# 3. Top 5 municipalities on each component
for pc in ["PC1", "PC2", "PC3"]:
    print(pc, gdf.sort_values(pc, ascending=False)["name"].head(5).tolist())

# 4. Map PC1
px.choropleth_map(gdf, geojson=gdf.geometry, locations=gdf.index,
                  color="PC1", hover_name="name",
                  color_continuous_scale="Reds",
                  center={"lat": -16.3, "lon": -64}, zoom=4,
                  title="PC1: Fire and forest-loss pressure (~2024)").show()



In [ ]:
from shapely.geometry import MultiPolygon
from shapely.geometry.polygon import orient

def make_clockwise(g):
    if g.geom_type == "Polygon":
        return orient(g, sign=-1.0)
    return MultiPolygon([orient(p, sign=-1.0) for p in g.geoms])

gdf["geometry"] = gdf.geometry.apply(make_clockwise)

In [ ]:
import plotly.graph_objects as go

def bolivia_map(gdf, col, title, scale="Reds"):
    data = gdf.dropna(subset=[col])                      # municipalities with a score

    fig = px.choropleth(data, geojson=gdf.geometry, locations=data.index,
                        color=col, hover_name="name",
                        color_continuous_scale=scale, title=title)

    # Grey base layer: all 343 municipalities (shows the ones without data)
    base = go.Choropleth(geojson=gdf.geometry.__geo_interface__,
                         locations=gdf.index, z=[0] * len(gdf),
                         colorscale=[[0, "lightgrey"], [1, "lightgrey"]],
                         showscale=False, hoverinfo="skip",
                         marker_line_color="white", marker_line_width=0.3)
    fig.add_trace(base)
    fig.data = fig.data[::-1]                            # draw grey first, colours on top

    fig.update_geos(fitbounds="locations", visible=False)   # zoom to Bolivia only
    fig.update_layout(height=700, margin=dict(l=0, r=0, t=50, b=0))
    fig.show()

bolivia_map(gdf, "PC1", "PC1: Fire and forest-loss pressure (~2024)")
bolivia_map(gdf, "PC2", "PC2 (~2024)", scale="Blues")
bolivia_map(gdf, "PC3", "PC3 (~2024)", scale="Greens")

# Part B

## Step B1: Calculate the changes

Earlier we used pivot for one variable. With 9 variables there's a simpler way: make one table for 2012 and one for 2024, then subtract.

In [ ]:
comp = ["sdg13_1_fec", "sdg13_1_icc", "sdg13_2_epb", "sdg13_2_ect", "sdg13_2_eci",
        "sdg15_1_sap", "sdg15_2_tpb", "sdg15_5_pbd", "sdg15_5_pbi"]

d12 = both[both["year"] == 2012].set_index("geo_id")[comp]
d24 = both[both["year"] == 2024].set_index("geo_id")[comp]

ch = d24 - d12            # change, 2012 -> 2024

print(ch.shape)
print(ch.describe().round(2).T)

# Share of municipalities with NO change at all
print((ch == 0).mean().round(2))

## Step B2: Correlation of the changes



In [ ]:
cols13 = ["sdg13_1_fec", "sdg13_1_icc", "sdg13_2_epb", "sdg13_2_ect", "sdg13_2_eci"]
cols15 = ["sdg15_1_sap", "sdg15_2_tpb", "sdg15_5_pbd", "sdg15_5_pbi"]

corr_ch = ch.corr(method="spearman").loc[cols13, cols15].round(2)
print(corr_ch)

px.imshow(corr_ch, text_auto=True, color_continuous_scale="RdBu",
          zmin=-1, zmax=1, aspect="auto",
          title="Spearman correlation of CHANGES 2012→2024: SDG 13 (rows) vs SDG 15 (columns)").show()

## Step B3: PCA on the changes

In [ ]:
d_ch = ch.dropna()                    # municipalities with all 9 changes (forest units)
print("Municipalities used:", len(d_ch))

Zc = StandardScaler().fit_transform(d_ch.rank())
pca_c = PCA().fit(Zc)

print(pd.Series(pca_c.explained_variance_ratio_.round(2),
                index=[f"PC{i+1}" for i in range(9)]))

loadings_c = pd.DataFrame(pca_c.components_[:3].T, index=comp,
                          columns=["PC1", "PC2", "PC3"]).round(2)

px.imshow(loadings_c.rename(index=labels), text_auto=True,
          color_continuous_scale="RdBu", zmin=-0.7, zmax=0.7, aspect="auto",
          title="PCA loadings of CHANGES 2012→2024 (rank-based)").show()

## Step B4: Map the change patterns

In [ ]:
# Scores of each municipality on the change components
scores_c = pd.DataFrame(pca_c.transform(Zc)[:, :3],
                        columns=["cPC1", "cPC2", "cPC3"],
                        index=d_ch.index).round(2)

# Attach to the map table (drop old versions first, in case you rerun)
gdf = gdf.drop(columns=["cPC1", "cPC2", "cPC3"], errors="ignore").join(scores_c)

for pc in ["cPC1", "cPC2", "cPC3"]:
    print(pc, gdf.sort_values(pc, ascending=False)["name"].head(5).tolist())

bolivia_map(gdf, "cPC1", "Change PC1: Deforestation intensification 2012→2024", scale="Reds")
bolivia_map(gdf, "cPC2", "Change PC2: Fires in natural areas vs disasters", scale="Purples")

## Step B5: Levels vs changes, a typology of municipalities

is high pressure in 2024 related to rising pressure since 2012? Plotting the level score (PC1) against the change score (cPC1) splits municipalities into four types:

In [ ]:
t = gdf.dropna(subset=["PC1", "cPC1"]).copy()

# Classify into 4 quadrants
t["type"] = "Low & falling"
t.loc[(t["PC1"] > 0) & (t["cPC1"] > 0), "type"] = "Hotspot: high & rising"
t.loc[(t["PC1"] <= 0) & (t["cPC1"] > 0), "type"] = "New frontier: low but rising"
t.loc[(t["PC1"] > 0) & (t["cPC1"] <= 0), "type"] = "Old frontier: high but slowing"

print(t["type"].value_counts())
print("Correlation PC1 vs cPC1:", round(t["PC1"].corr(t["cPC1"]), 2))

for k in ["New frontier: low but rising", "Old frontier: high but slowing"]:
    print(k, t[t["type"] == k].sort_values("cPC1", ascending=(k.startswith("Old")))["name"].head(6).tolist())

fig = px.scatter(t, x="PC1", y="cPC1", color="type", hover_name="name",
                 labels={"PC1": "Level 2024 (PC1: fire & forest loss)",
                         "cPC1": "Change 2012→2024 (cPC1)"},
                 title="Fire & deforestation: level vs change")
fig.add_hline(y=0, line_dash="dash", line_color="grey")
fig.add_vline(x=0, line_dash="dash", line_color="grey")
fig.show()